# INFO 7390 — Advanced Data Visualization Teaching Notebook

**Project:** Advanced Data Visualization — Interactive dashboards, perception-based design, and storytelling with data  
**Author:** Mubin Modi
**Course:** INFO 7390: Advanced Data Science and Architecture  
---

## Overview

This repository contains a teaching notebook and supporting assets to learn and practice advanced data visualization concepts. The notebook demonstrates perception-driven visualization design, interactive exploration, and storytelling patterns using Python (Pandas + Plotly) inside a Jupyter Notebook.

Primary learning goals:
- Choose perceptually effective visual encodings for different data types.
- Build interactive visualizations for in-notebook exploration.
- Create a short data-driven narrative (annotated visuals + guided story panel).

---

## Contents



In [14]:
# Setup & imports
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from google.colab import output
output.enable_custom_widget_manager()
import ipywidgets as widgets
from ipywidgets import interact
# For interactivity in Jupyter
try:
    import ipywidgets as widgets
    from ipywidgets import interact, interactive, fixed, HBox, VBox
except Exception as e:
    print("ipywidgets not found. Install with `pip install ipywidgets` to enable in-notebook widgets.")

# Display settings
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 200)


In [15]:
# Data loading: try common locations, else fallback to plotly gapminder sample
def load_data():
    # 1) try to load local CSV in /data/gapminder.csv
    import os
    candidates = [
        "data/gapminder.csv",
        "./data/gapminder.csv",
        "/mnt/data/gapminder.csv",
    ]
    for path in candidates:
        if os.path.exists(path):
            print(f"Loading dataset from {path}")
            return pd.read_csv(path)

    # 2) try to load a dataset shipped in your notebook (if present)
    # NOTE: If your existing notebook uses a different dataset, replace or map columns as required.

    # 3) fallback to Plotly's built-in gapminder
    try:
        df = px.data.gapminder()
        print("Loaded plotly gapminder dataset as fallback.")
        return df
    except Exception as e:
        raise FileNotFoundError("No dataset found. Please add `data/gapminder.csv` or modify the loader.")

df = load_data()
df.head()


Loaded plotly gapminder dataset as fallback.


,country,continent,year,lifeExp,pop,gdpPercap,iso_alpha,iso_num
0,Afghanistan,Asia,1952,28.801,8425333,779.445314,AFG,4
1,Afghanistan,Asia,1957,30.332,9240934,820.853030,AFG,4
2,Afghanistan,Asia,1962,31.997,10267083,853.100710,AFG,4
3,Afghanistan,Asia,1967,34.020,11537966,836.197138,AFG,4
4,Afghanistan,Asia,1972,36.088,13079460,739.981106,AFG,4


## Quick data inspection

We expect (or will map to) columns: `country`, `continent`, `year`, `lifeExp`, `pop`, `gdpPercap`.
If your dataset uses different names, edit the mapping below.


In [16]:
# Inspect columns and sample
display(df.dtypes)
display(df.head())

# Optional: map alternative names - change mapping here if needed
col_map = {}
# e.g., col_map = {"CountryName": "country", "GDP": "gdpPercap"}
if col_map:
    df = df.rename(columns=col_map)
    print("Applied user column mapping.")


,0
country,object
continent,object
year,int64
lifeExp,float64
pop,int64
gdpPercap,float64
iso_alpha,object
iso_num,int64


,country,continent,year,lifeExp,pop,gdpPercap,iso_alpha,iso_num
0,Afghanistan,Asia,1952,28.801,8425333,779.445314,AFG,4
1,Afghanistan,Asia,1957,30.332,9240934,820.853030,AFG,4
2,Afghanistan,Asia,1962,31.997,10267083,853.100710,AFG,4
3,Afghanistan,Asia,1967,34.020,11537966,836.197138,AFG,4
4,Afghanistan,Asia,1972,36.088,13079460,739.981106,AFG,4


## Scatter plot (perception-aware): GDP per capita (log scale) vs Life expectancy

Rationale:
- GDP per capita is heavily right-skewed; using a log scale for the x-axis makes the distribution more legible.
- Size encodes population (use area encoding properly).
- Keep chart junk minimal and include concise axis labels and tooltips.


In [17]:
# Choose a year to display
year = df['year'].max()
filtered = df[df.year == year]

fig = px.scatter(
    filtered,
    x="gdpPercap",
    y="lifeExp",
    size="pop",
    color="continent",
    hover_name="country",
    log_x=True,
    size_max=60,
    labels={"gdpPercap": "GDP per capita (log scale)", "lifeExp": "Life expectancy"}
)
fig.update_layout(template="simple_white", title=f"GDP per capita vs Life expectancy — {year}",
                  margin=dict(l=30, r=10, t=40, b=30))
fig.show()


## Bar chart: Ranking top countries for a metric

Rationale:
- Use horizontal bar charts to show ranking (position is perceived more accurately than area or color).
- Use sorts and limit to top-k to avoid clutter.


In [18]:
# Top 15 countries by life expectancy in selected year
top_k = 15
metric = "lifeExp"
top = filtered.sort_values(metric, ascending=False).head(top_k)
fig = px.bar(
    top[::-1],  # reverse to show top at top when horizontal
    x=metric,
    y="country",
    orientation="h",
    labels={metric: "Life Expectancy", "country": "Country"},
    height=500
)
fig.update_layout(template="simple_white", title=f"Top {top_k} countries by Life Expectancy — {year}")
fig.show()


## Small multiples / line charts: Trends over time (median per continent)

Rationale:
- Use small multiples to compare similar charts across categories; each small chart uses the same scale for comparability.
- Here we show median of the chosen metric across years per continent.


In [19]:
metric = "gdpPercap"  # choose metric to analyze over time
median_ts = df.groupby(["year", "continent"])[metric].median().reset_index()

fig = px.line(median_ts, x="year", y=metric, color="continent",
              labels={metric: metric, "year": "Year"}, height=400)
fig.update_layout(template="simple_white", title=f"Median {metric} by continent (over time)")
fig.show()


## In-notebook interactivity (widgets)

This section creates interactive controls for exploring the dataset within the notebook.  
Use the dropdowns and slider to change year/metric/continents and see how charts update.


In [20]:
metrics = ["gdpPercap", "lifeExp", "pop"]
years = sorted(df.year.unique())
continents = sorted(df.continent.unique())

@interact(
    metric = widgets.Dropdown(options=metrics, value="lifeExp", description="Metric:"),
    year = widgets.IntSlider(min=min(years), max=max(years), step=1, value=max(years), description="Year:"),
    continent = widgets.SelectMultiple(options=continents, value=tuple(continents), description="Continents:")
)
def explore(metric, year, continent):
    filtered = df[(df.year == year) & (df.continent.isin(continent))]

    if metric == "gdpPercap":
        fig = px.scatter(
            filtered,
            x="gdpPercap",
            y="lifeExp",
            size="pop",
            color="continent",
            hover_name="country",
            log_x=True,
            size_max=50
        )
    else:
        fig = px.scatter(
            filtered,
            x="gdpPercap",
            y=metric,
            size="pop",
            color="continent",
            hover_name="country",
            log_x=True,
            size_max=50
        )

    fig.update_layout(
        template="simple_white",
        height=450,
        margin=dict(l=20, r=20, t=30, b=20)
    )

    fig.show()


interactive(children=(Dropdown(description='Metric:', index=1, options=('gdpPercap', 'lifeExp', 'pop'), value=…

## Storytelling: Guided annotations and narrative panel

Approach:
- Provide a small "story panel" that highlights a few notable observations (top/bottom performers, outliers).
- Use annotations on the chart to call out key points.


In [21]:
# Simple storytelling function
def story_panel(year, metric="gdpPercap", top_n=3):
    filtered = df[df.year == year].copy()
    if metric not in filtered.columns:
        raise ValueError(f"{metric} not found in data columns.")
    # Top and bottom
    top = filtered.nlargest(top_n, metric)
    bottom = filtered.nsmallest(top_n, metric)
    display(Markdown(f"### Story highlights — {year} — metric: **{metric}**"))
    display(Markdown("**Top performers:**"))
    display(top[["country", "continent", metric]].reset_index(drop=True))
    display(Markdown("**Lowest performers:**"))
    display(bottom[["country", "continent", metric]].reset_index(drop=True))

    # Annotated scatter (annotate top1)
    fig = px.scatter(filtered, x="gdpPercap", y="lifeExp", hover_name="country", log_x=True,
                     color="continent", size="pop", size_max=50)
    if not top.empty:
        top1 = top.iloc[0]
        fig.add_annotation(
            x=top1["gdpPercap"],
            y=top1["lifeExp"],
            text=f"{top1['country']} (Top {metric})",
            showarrow=True,
            arrowhead=2
        )
    fig.update_layout(template="simple_white", title=f"Annotated scatter — {year}")
    fig.show()

# Example usage:
story_panel(year=df.year.max(), metric="gdpPercap", top_n=3)


### Story highlights — 2007 — metric: **gdpPercap**

**Top performers:**

,country,continent,gdpPercap
0,Norway,Europe,49357.19017
1,Kuwait,Asia,47306.98978
2,Singapore,Asia,47143.17964


**Lowest performers:**

,country,continent,gdpPercap
0,"Congo, Dem. Rep.",Africa,277.551859
1,Liberia,Africa,414.507341
2,Burundi,Africa,430.070692


## Exercises (Try)

**Exercise 1 (Beginner)**  
Create a horizontal bar chart of the top 10 countries by population for year = 2007. Explain why a horizontal bar is a good choice.

**Exercise 2 (Intermediate)**  
Create an interactive widget to animate the distribution of `gdpPercap` over years using either Plotly's `animation_frame` or ipywidgets. Discuss one pitfall of animations for data interpretation.

**Exercise 3 (Advanced)**  
Design a short 3-bullet narrative (2–3 sentences total) that explains a surprising pattern you see in the data when comparing GDP per capita and life expectancy. Use an annotated chart to support your narrative.

When you finish, run the solution cell below to check.


In [22]:
# Solution cell (run to reveal solutions)
print("=== Exercise 1 solution (code) ===")
sol1 = filtered = df[(df.year == 2007)].sort_values("pop", ascending=False).head(10)
fig1 = px.bar(sol1[::-1], x="pop", y="country", orientation="h", labels={"pop":"Population"}, height=450)
fig1.update_layout(template="simple_white", title="Top 10 countries by population — 2007")
fig1.show()
print("\nRationale: Horizontal bars make reading country names and comparing position easy; position encodes magnitude accurately.")

print("\n=== Exercise 2 solution (approach) ===")
print("Option A: Plotly animation_frame: px.scatter(..., animation_frame='year')")
print("Option B: ipywidgets slider with lambda to update plot per year.")
print("Pitfall: Rapid animations can hide important changes and make comparisons across frames difficult.")

print("\n=== Exercise 3 suggestion ===")
display(Markdown("**Narrative example:** Despite high GDP per capita in a few small countries, life expectancy gains are often more gradual across continents — showing that wealth alone doesn't instantly translate to health outcomes. See annotated scatter above for 2007 highlighting top GDP countries."))


=== Exercise 1 solution (code) ===



Rationale: Horizontal bars make reading country names and comparing position easy; position encodes magnitude accurately.

=== Exercise 2 solution (approach) ===
Option A: Plotly animation_frame: px.scatter(..., animation_frame='year')
Option B: ipywidgets slider with lambda to update plot per year.
Pitfall: Rapid animations can hide important changes and make comparisons across frames difficult.

=== Exercise 3 suggestion ===


**Narrative example:** Despite high GDP per capita in a few small countries, life expectancy gains are often more gradual across continents — showing that wealth alone doesn't instantly translate to health outcomes. See annotated scatter above for 2007 highlighting top GDP countries.

## Export & packaging tips

- Export the notebook as PDF for the Canvas submission: `File -> Download as -> PDF` (or use `nbconvert`).
- Include a professional `README.md`, `requirements.txt`, and sample data in `/data`.
- If you want a deployable app, add the Streamlit starter file below in `app/streamlit_app.py` and run `streamlit run app/streamlit_app.py`.


#Conclusion


In this notebook, we explored advanced data visualization techniques by combining perception-driven design, storytelling principles, and interactive exploration using plotly and ipywidgets.
We demonstrated how interactivity transforms traditional static charts into powerful analytical tools, empowering users to ask their own questions and uncover insights dynamically.

By walking through the creation of an interactive scatter visualization—complete with filtering, dynamic metrics, and real-time updates—we showed how visualization is not just a presentation tool, but an essential component of modern data science workflows.
Interactive dashboards help bridge the gap between technical analyses and stakeholder understanding, making insights more accessible, intuitive, and actionable.

This notebook serves as both a learning resource and a template for building more advanced dashboards, such as:

Multi-page analytical apps

Time-series visual monitors

Geo-spatial visualizations

Story-driven narrative reports

Model interpretability dashboards

Ultimately, effective visualization is about communication—turning data into understanding.
You now have the foundation to design thoughtful, perception-aware, and interactive visual systems that elevate the impact of your data science work.

#License

MIT License

Copyright (c) 2025 Mubin Modi

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in
all copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN
THE SOFTWARE.


#References

Bostock, M., Ogievetsky, V., & Heer, J. (2011).
D³ Data-Driven Documents. IEEE Transactions on Visualization and Computer Graphics, 17(12), 2301–2309.

Heer, J., Bostock, M., & Ogievetsky, V. (2010).
A Tour Through the Visualization Zoo. Communications of the ACM, 53(6), 59–67.

Few, S. (2012).
Show Me the Numbers: Designing Tables and Graphs to Enlighten. Analytics Press.

Tufte, E. R. (2001).
The Visual Display of Quantitative Information. Graphics Press.

McKinney, W. (2017).
Python for Data Analysis. O’Reilly Media.

Plotly Technologies Inc. (2015).
Collaborative data science. Retrieved from https://plot.ly

Jupyter Project. (2023).
ipywidgets Documentation. https://ipywidgets.readthedocs.io/

Gapminder Foundation. (2023).
Gapminder Data Documentation. https://www.gapminder.org/data/

Shneiderman, B. (1996).
The Eyes Have It: A Task by Data Type Taxonomy for Information Visualizations. IEEE Symposium on Visual Languages.